# Bagging by hand
Three decision trees, each trained on its own bootstrap sample of a tiny iris training set, then a majority vote.
Then the four ways of sampling: bagging, pasting, random subspaces, random patches.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier, export_text

## 1. The data: two overlapping species, two columns

In [2]:
iris = load_iris(as_frame=True).frame.rename(columns={"target": "species"})
# Keep versicolor (1) and virginica (2), and two columns
df = iris[iris["species"] != 0][["sepal width (cm)", "petal length (cm)", "species"]]
print(df.shape)

# Shuffle the 100 rows (they are stored species by species)
df = df.sample(100, random_state=2)

# Tiny sets, so every step can be printed: 10 training rows, 5 validation rows, 5 test rows
df_train = df.iloc[:60].sample(10, random_state=2)
df_val = df.iloc[60:80].sample(5, random_state=2)
df_test = df.iloc[80:].sample(5, random_state=2)
X_val, y_val = df_val.iloc[:, 0:2], df_val.iloc[:, -1]
df_train

(100, 3)


,sepal width (cm),petal length (cm),species
56,3.3,4.7,1
133,2.8,5.1,2
76,2.8,4.8,1
91,3.0,4.6,1
94,2.7,4.2,1
69,2.5,3.9,1
128,2.8,5.6,2
124,3.3,5.7,2
105,3.0,6.6,2
68,2.2,4.5,1


## 2. Bootstrapping: three trees, three samples
Each sample draws 8 rows from the 10 training rows **with replacement** (`replace=True`), so rows can repeat.

In [3]:
trees = []
for b in range(3):
    df_bag = df_train.sample(8, replace=True, random_state=b)
    print(f"--- sample for tree {b + 1}: row labels {sorted(df_bag.index.tolist())}")
    # max_depth is not set, so each tree grows fully: low bias, high variance
    tree = DecisionTreeClassifier(random_state=0).fit(df_bag.iloc[:, 0:2], df_bag.iloc[:, -1])
    print(export_text(tree, feature_names=["sepal width", "petal length"]))
    print("validation accuracy", accuracy_score(y_val, tree.predict(X_val)))
    trees.append(tree)

--- sample for tree 1: row labels [56, 68, 69, 69, 91, 91, 91, 124]


|--- petal length <= 5.20
|   |--- class: 1
|--- petal length >  5.20
|   |--- class: 2

validation accuracy 0.4
--- sample for tree 2: row labels [56, 56, 68, 69, 69, 105, 124, 133]
|--- petal length <= 4.90
|   |--- class: 1
|--- petal length >  4.90
|   |--- class: 2

validation accuracy 0.8
--- sample for tree 3: row labels [76, 76, 105, 105, 105, 124, 128, 133]
|--- petal length <= 4.95
|   |--- class: 1
|--- petal length >  4.95
|   |--- class: 2

validation accuracy 0.6


## 3. Aggregation: majority vote on the test rows

In [4]:
votes = pd.DataFrame({f"tree {i + 1}": t.predict(df_test.iloc[:, 0:2]) for i, t in enumerate(trees)}, index=df_test.index)
votes["majority"] = votes.mode(axis=1)[0].astype(int)
votes["true"] = df_test["species"]
pd.concat([df_test.iloc[:, 0:2], votes], axis=1)

,sepal width (cm),petal length (cm),tree 1,tree 2,tree 3,majority,true
57,2.4,3.3,1,1,1,1,1
70,3.2,4.8,1,1,1,1,1
65,3.1,4.4,1,1,1,1,1
54,2.8,4.6,1,1,1,1,1
143,3.2,5.9,2,2,2,2,2


On these five rows all three trees agree. A row where they do not: iris row 119, a virginica (class 2) with
sepal width 2.2 and petal length 5.0.

In [5]:
point = pd.DataFrame([[2.2, 5.0]], columns=df.columns[:2])
for i, t in enumerate(trees):
    print(f"tree {i + 1} predicts", t.predict(point)[0])

tree 1 predicts 1
tree 2 predicts 2
tree 3 predicts 2


## 4. Four ways to sample

In [6]:
# Pasting: rows without replacement (replace=False is the default), so no row repeats
df_train.sample(8, random_state=0)

,sepal width (cm),petal length (cm),species
76,2.8,4.8,1
105,3.0,6.6,2
94,2.7,4.2,1
68,2.2,4.5,1
133,2.8,5.1,2
128,2.8,5.6,2
124,3.3,5.7,2
91,3.0,4.6,1


In [7]:
# Random subspaces: all rows, a random choice of columns (axis=1 samples columns)
df10 = load_iris(as_frame=True).frame.sample(10, random_state=0)
df10.drop(columns="target").sample(2, replace=True, axis=1, random_state=1)

,sepal width (cm),petal width (cm)
114,2.8,2.4
62,2.2,1.0
33,4.2,0.2
107,2.9,1.8
7,3.4,0.2
100,3.3,2.5
40,3.5,0.3
86,3.1,1.5
76,2.8,1.4
71,2.8,1.3


In [8]:
# Random patches: sample rows, then columns
df10.drop(columns="target").sample(8, replace=True, random_state=0).sample(2, replace=True, axis=1, random_state=3)

,petal length (cm),sepal length (cm)
100,6.0,6.3
114,5.1,5.8
107,6.3,7.3
107,6.3,7.3
86,4.7,6.7
71,4.0,6.1
107,6.3,7.3
100,6.0,6.3


## 5. How many different rows does a bootstrap sample hold?
Draw n rows from n with replacement, many times, and count the distinct ones.

In [9]:
rng = np.random.default_rng(0)
for n in [10, 100, 1000, 10000]:
    share = np.mean([len(np.unique(rng.integers(0, n, n))) / n for _ in range(2000)])
    print(f"n = {n:>6}: simulated share of distinct rows {share:.3f}, formula 1 - (1 - 1/n)^n = {1 - (1 - 1/n) ** n:.3f}")
print("limit 1 - 1/e =", round(1 - np.exp(-1), 4))

n =     10: simulated share of distinct rows 0.655, formula 1 - (1 - 1/n)^n = 0.651
n =    100: simulated share of distinct rows 0.635, formula 1 - (1 - 1/n)^n = 0.634


n =   1000: simulated share of distinct rows 0.632, formula 1 - (1 - 1/n)^n = 0.632


n =  10000: simulated share of distinct rows 0.632, formula 1 - (1 - 1/n)^n = 0.632
limit 1 - 1/e = 0.6321
